# Phase 6: Model Tuning (Hyperparameter Tuning) & Validation

In [1]:
# Cell 1: Setup for Hyperparameter Tuning
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report, roc_curve, auc
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings('ignore')

# Load the data
X_train = pd.read_csv('../data/X_train.csv')
X_test = pd.read_csv('../data/X_test.csv')
y_train = pd.read_csv('../data/y_train.csv').squeeze()
y_test = pd.read_csv('../data/y_test.csv').squeeze()

print("Data loaded! Ready for tuning.")

Data loaded! Ready for tuning.


In [3]:
# Cell 2: Define Hyperparameters & Run RandomizedSearchCV
print("Starting Randomized Search... (This will take a few minutes)")

# 1. Create the base model
rf_base = RandomForestClassifier(random_state=42)

# 2. Define the hyperparameter grid (options to test)
random_grid = {
    'n_estimators': [50, 100, 200],           # Number of trees
    'max_depth': [10, 20, 30, None],          # Maximum depth of the tree
    'min_samples_split': [2, 5, 10],          # Min samples required to split a node
    'min_samples_leaf': [1, 2, 4]             # Min samples required at each leaf node
}

# 3. Setup RandomizedSearchCV (n_iter=10 means it will try 10 random combinations)
rf_random = RandomizedSearchCV(estimator=rf_base, 
                               param_distributions=random_grid, 
                               n_iter=10, 
                               cv=3, # 3-fold cross validation for speed
                               verbose=2, 
                               random_state=42, 
                               n_jobs=-1) # Uses all CPU cores

# 4. Fit the random search model
rf_random.fit(X_train, y_train)

print("\n✅ Tuning Completed!")
print(f"Best Parameters Found: {rf_random.best_params_}")

Starting Randomized Search... (This will take a few minutes)
Fitting 3 folds for each of 10 candidates, totalling 30 fits

✅ Tuning Completed!
Best Parameters Found: {'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_depth': 30}


In [4]:
# Cell 3: Evaluate the Best Model
best_rf_model = rf_random.best_estimator_

# Predict on test data
y_pred_tuned = best_rf_model.predict(X_test)

# Print metrics
tuned_accuracy = accuracy_score(y_test, y_pred_tuned)
print(f"Tuned Model Accuracy: {tuned_accuracy * 100:.2f}%\n")
print("Tuned Classification Report:")
print(classification_report(y_test, y_pred_tuned))

# ROC-AUC Validation
y_pred_prob_tuned = best_rf_model.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob_tuned)
roc_auc = auc(fpr, tpr)
print(f"Tuned Model AUC Score: {roc_auc:.2f}")

Tuned Model Accuracy: 50.76%

Tuned Classification Report:
              precision    recall  f1-score   support

           0       0.51      0.50      0.51     20359
           1       0.50      0.51      0.51     20043

    accuracy                           0.51     40402
   macro avg       0.51      0.51      0.51     40402
weighted avg       0.51      0.51      0.51     40402

Tuned Model AUC Score: 0.51


In [5]:
# Cell 4: Save the Tuned Model
model_path = '../src/random_forest_model_tuned.pkl'
joblib.dump(best_rf_model, model_path)

print(f"🎉 Best Tuned Model saved successfully to {model_path}!")

🎉 Best Tuned Model saved successfully to ../src/random_forest_model_tuned.pkl!
